# NN Verification Fundamentals

## Historical Context: Reluplex (Narrative)
Reluplex (Katz et al., 2017) was the first scalable neural network verifier. It extended the Simplex algorithm (linear programming) to handle ReLU activations by:
1. Treating ReLU outputs as new variables with linear constraints
2. Using the Simplex method to propagate bounds
3. When a ReLU constraint is violated, splitting into two cases: neuron active vs inactive

Reluplex was complete (finds all counterexamples) but had exponential worst-case complexity in the number of neurons. It verified properties on small ACAS Xu (aircraft collision avoidance) networks with 300 neurons in minutes to hours.

## Completeness vs Scalability Tradeoff
- **Complete methods** (Reluplex, Marabou, α,β-CROWN exact): Find all counterexamples but exponential worst case
- **Sound over-approximations** (CROWN, DeepPoly, IBP): Always correct when they say CERTIFIED, but may say UNKNOWN when system is actually safe (sound but incomplete)
- **Empirical methods** (PGD, AutoAttack): Find counterexamples but can't prove safety

In [ ]:
import numpy as np
from typing import Tuple, List, Dict, Optional

# Implement naive branch-and-bound NN verification
# This captures the core idea of Reluplex/complete verification

np.random.seed(42)

class SmallReLUNet:
    def __init__(self, W1, b1, W2, b2):
        self.W1 = W1; self.b1 = b1
        self.W2 = W2; self.b2 = b2
    
    def forward(self, x):
        return self.W2 @ np.maximum(0, self.W1 @ x + self.b1) + self.b2

# Property: for all x in input_box, output[0] > output[1]
# (i.e., class 0 always wins)

def interval_bound_propagation(net, x_lo, x_hi):
    """Compute output bounds via interval arithmetic."""
    W1p = np.maximum(0, net.W1)
    W1n = np.minimum(0, net.W1)
    pre1_lo = W1p @ x_lo + W1n @ x_hi + net.b1
    pre1_hi = W1p @ x_hi + W1n @ x_lo + net.b1
    h_lo = np.maximum(0, pre1_lo)
    h_hi = np.maximum(0, pre1_hi)
    W2p = np.maximum(0, net.W2)
    W2n = np.minimum(0, net.W2)
    out_lo = W2p @ h_lo + W2n @ h_hi + net.b2
    out_hi = W2p @ h_hi + W2n @ h_lo + net.b2
    return out_lo, out_hi, pre1_lo, pre1_hi

def verify_robustness(net, x0, eps, class_idx=0, max_splits=100):
    """
    Verify that net classifies all inputs in L-inf(x0, eps) as class_idx.
    Uses branch-and-bound: split when neuron is in ambiguous range.
    """
    queue = [(x0 - eps, x0 + eps)]
    n_splits = 0
    
    while queue and n_splits < max_splits:
        x_lo, x_hi = queue.pop()
        out_lo, out_hi, pre1_lo, pre1_hi = interval_bound_propagation(net, x_lo, x_hi)
        
        n_classes = len(out_lo)
        # Check if class_idx is certified dominant
        other_classes = [i for i in range(n_classes) if i != class_idx]
        
        if all(out_lo[class_idx] > out_hi[j] for j in other_classes):
            continue  # certified in this sub-box
        
        if any(out_hi[class_idx] < out_lo[j] for j in other_classes):
            return 'VIOLATED', x_lo, x_hi  # counterexample region found
        
        # Ambiguous: split on the most uncertain ReLU neuron
        ambiguous = [(pre1_lo[i], pre1_hi[i], i) 
                     for i in range(len(pre1_lo))
                     if pre1_lo[i] < 0 < pre1_hi[i]]
        
        if not ambiguous:
            continue
        
        # Split on the neuron with most uncertainty
        _, _, split_neuron = max(ambiguous, key=lambda t: t[1] - t[0])
        
        # Find which input dimension most influences this neuron
        influence = np.abs(net.W1[split_neuron])
        split_dim = np.argmax(influence)
        mid = (x_lo[split_dim] + x_hi[split_dim]) / 2
        
        x_lo1, x_hi1 = x_lo.copy(), x_hi.copy()
        x_lo2, x_hi2 = x_lo.copy(), x_hi.copy()
        x_hi1[split_dim] = mid
        x_lo2[split_dim] = mid
        
        queue.extend([(x_lo1, x_hi1), (x_lo2, x_hi2)])
        n_splits += 1
    
    if n_splits >= max_splits:
        return 'UNKNOWN', None, None
    return 'CERTIFIED', None, None

# Build and test the verifier
net = SmallReLUNet(
    W1=np.array([[2.0, 1.0], [-1.0, 2.0], [1.5, -0.5]]),
    b1=np.array([0.5, 0.5, 0.2]),
    W2=np.array([[1.0, 0.5, 0.3], [-1.0, -0.5, -0.3]]),
    b2=np.array([0.1, -0.1])
)

x0 = np.array([0.5, 0.3])
out0 = net.forward(x0)
class0 = np.argmax(out0)
print(f'Network output at x0: {out0}')
print(f'Class: {class0}')

for eps in [0.05, 0.1, 0.3]:
    result, viol_lo, viol_hi = verify_robustness(net, x0, eps, class_idx=class0)
    print(f'Verification eps={eps}: {result}')
